In [6]:
#1D MATRIX ADDITION
from numba import cuda
import numpy as np

@cuda.jit
def add_1d(a, b, c):
    i = cuda.grid(1)         
    if i < c.size:            
        c[i] = a[i] + b[i]
N = 10                       
a = np.arange(N, dtype=np.int32)
b = np.arange(N, dtype=np.int32) * 10
c = np.zeros(N, dtype=np.int32)

d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_c = cuda.to_device(c)

threads_per_block = 128
blocks = (N + threads_per_block - 1) // threads_per_block

add_1d[blocks, threads_per_block](d_a, d_b, d_c)
cuda.synchronize()

result = d_c.copy_to_host()

print("A:", a)
print("B:", b)
print("C = A + B:", result)

A: [0 1 2 3 4 5 6 7 8 9]
B: [ 0 10 20 30 40 50 60 70 80 90]
C = A + B: [ 0 11 22 33 44 55 66 77 88 99]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [7]:
#2D MATRIX ADDITION
from numba import cuda
import numpy as np

@cuda.jit
def add_2d(a, b, c):
    row, col = cuda.grid(2)    

    if row < c.shape[0] and col < c.shape[1]:
        c[row, col] = a[row, col] + b[row, col]

rows, cols = 4, 5

A = np.arange(rows * cols, dtype=np.int32).reshape(rows, cols)
B = np.ones((rows, cols), dtype=np.int32) * 10
C = np.zeros((rows, cols), dtype=np.int32)

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.to_device(C)

threads_per_block = (16, 16)
blocks_per_grid_x = (cols + threads_per_block[1] - 1) // threads_per_block[1]
blocks_per_grid_y = (rows + threads_per_block[0] - 1) // threads_per_block[0]
blocks = (blocks_per_grid_y, blocks_per_grid_x)


add_2d[blocks, threads_per_block](d_A, d_B, d_C)
cuda.synchronize()

result = d_C.copy_to_host()

print("Matrix A:\n", A)
print("Matrix B:\n", B)
print("Matrix C = A + B:\n", result)

Matrix A:
 [[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]
Matrix B:
 [[10 10 10 10 10]
 [10 10 10 10 10]
 [10 10 10 10 10]
 [10 10 10 10 10]]
Matrix C = A + B:
 [[10 11 12 13 14]
 [15 16 17 18 19]
 [20 21 22 23 24]
 [25 26 27 28 29]]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
